In [2]:
import pandas as pd


## Baseball
Being avid baseball fans and quantitative analysts, we are rather unhappy 
with the manner in which baseball statistics are calculated. Recording 
the absolute number of hits and homeruns in a season seems like a very 
poor way to compare players over time. What we really want is to calculate 
some relative statistics! You’ll find the information on baseball statistic 
abbreviations at http://mlb.mlb.com/mlb/official_info/baseball_basics/abbreviations.jsp 
helpful for this project. Brief column descriptions are also available in 
"Data set information.docx".

1. Compare the top five players w.r.t.  the maximum homeruns to the top five players 
w.r.t. homeruns as a percentage of at bats

First, we need to read the data.

In [14]:
baseball = pd.read_csv("baseball.csv")


Then we'll take a quick look to get a feel for the data set

In [15]:
print(baseball.head())

   Unnamed: 0         id  year  stint team   lg   g   ab   r   h  ...   rbi  \
0           4  ansonca01  1871      1  RC1  NaN  25  120  29  39  ...  16.0   
1          44  forceda01  1871      1  WS3  NaN  32  162  45  45  ...  29.0   
2          68  mathebo01  1871      1  FW1  NaN  19   89  15  24  ...  10.0   
3          99  startjo01  1871      1  NY2  NaN  33  161  35  58  ...  34.0   
4         102  suttoez01  1871      1  CL1  NaN  29  128  35  45  ...  23.0   

    sb   cs  bb   so  ibb  hbp  sh  sf  gidp  
0  6.0  2.0   2  1.0  NaN  NaN NaN NaN   NaN  
1  8.0  0.0   4  0.0  NaN  NaN NaN NaN   NaN  
2  2.0  1.0   2  0.0  NaN  NaN NaN NaN   NaN  
3  4.0  2.0   3  0.0  NaN  NaN NaN NaN   NaN  
4  3.0  1.0   1  0.0  NaN  NaN NaN NaN   NaN  

[5 rows x 23 columns]


Next, to make it easy, I'll create a subset with only 2005 data.

In [16]:
bb2005 = baseball[baseball['year'] == 2005]


...and I'll get a sense for the resulting data frame

In [18]:
print(bb2005.head())
print(bb2005.shape)


       Unnamed: 0         id  year  stint team  lg    g   ab   r   h  ...  \
21378       87533  sprinru01  2005      1  HOU  NL   62    0   0   0  ...   
21379       87610  micelda01  2005      1  COL  NL   19    0   0   0  ...   
21380       87634  sweenma01  2005      1  SDN  NL  135  221  31  65  ...   
21381       87636  hanseda01  2005      1  SEA  AL   60   75   5  13  ...   
21382       87641  willige02  2005      1  NYN  NL   39   30   9   7  ...   

        rbi   sb   cs  bb    so  ibb  hbp   sh   sf  gidp  
21378   0.0  0.0  0.0   0   0.0  0.0  0.0  0.0  0.0   0.0  
21379   0.0  0.0  0.0   0   0.0  0.0  0.0  0.0  0.0   0.0  
21380  40.0  4.0  0.0  40  58.0  3.0  0.0  1.0  5.0   6.0  
21381  11.0  1.0  0.0   9  19.0  1.0  0.0  2.0  2.0   1.0  
21382   3.0  2.0  0.0   1   7.0  0.0  0.0  1.0  0.0   0.0  

[5 rows x 23 columns]
(160, 23)


First we'll add a hrpctab (for homeruns as a percentage of at bats)
column to the dataframe. Before we can calculate the percentage
we need to make sure that we won't have divide by zeros or
missing data

In [19]:
bb2005_clean = bb2005.dropna(subset=['hr','ab'])
print(bb2005_clean.shape)
print(bb2005.shape)


(160, 23)
(160, 23)


It appears there was no missing data (same number of rows before and after).
Now I'll eliminate any rows with ab <= 0 and drop all columns
except the three we need (id, hr, ab) to make the top 5 lists
easier to work with

In [20]:
bb2005_clean = (bb2005[bb2005['ab']>0])[['id','hr','ab']]
print(bb2005_clean.shape)


(123, 3)


   looks like we lost 37 players... now we can proceed with 
   calculating the homeruns as a percentage of at bats

In [21]:
bb2005_clean['hrpctab'] = bb2005_clean['hr'] / bb2005_clean['ab']
print(bb2005_clean.head())


              id  hr   ab   hrpctab
21380  sweenma01   8  221  0.036199
21381  hanseda01   2   75  0.026667
21382  willige02   1   30  0.033333
21383  harrile01   1   70  0.014286
21384  stairma01  13  396  0.032828


   The next step is to identify the top 5 players in terms 
   of number of home runs

In [22]:
top5hr = bb2005_clean.nlargest(5,'hr')
print(top5hr)


              id  hr   ab   hrpctab
21510  ramirma02  45  554  0.081227
21496  griffke02  35  491  0.071283
21480  floydcl01  34  550  0.061818
21523  sheffga01  34  584  0.058219
21388  delgaca01  33  521  0.063340


   ... and we can find the top 5 w.r.t. hr as % of ab

In [23]:
top5hrpctab = bb2005_clean.nlargest(5,'hrpctab')
print(top5hrpctab)


              id  hr   ab   hrpctab
21414  bondsba01   5   42  0.119048
21503  thomafr04  12  105  0.114286
21510  ramirma02  45  554  0.081227
21496  griffke02  35  491  0.071283
21412  sandere02  21  295  0.071186


   Now we'll merge them to create a comparison table:
   First step is toreplace the dataframe indexes so that the rows will align

In [24]:
top5hr.index = list(range(1,6))
top5hrpctab.index = list(range(1,6))


   Next, we concatenate the two dataframes. The keys will differentiate
   the columns between the absolute hr and hr as % ab groups

In [25]:
compare = pd.concat([top5hr, top5hrpctab], 
                    axis=1,
                    keys=['Absolute','Percentage'])
print(compare)


    Absolute                    Percentage                   
          id  hr   ab   hrpctab         id  hr   ab   hrpctab
1  ramirma02  45  554  0.081227  bondsba01   5   42  0.119048
2  griffke02  35  491  0.071283  thomafr04  12  105  0.114286
3  floydcl01  34  550  0.061818  ramirma02  45  554  0.081227
4  sheffga01  34  584  0.058219  griffke02  35  491  0.071283
5  delgaca01  33  521  0.063340  sandere02  21  295  0.071186


   We can see that these two methods of ranking performance yield a
   very different list of top 6 players. In fact, it appears that only
   two players, Manny Ramirez and Ken Griffey Jr., appear in both top 5 lists. 
   Moreover, we see that the top homerun hitter based on percentage of at bats,
   hits nearly 32% more home runs as a percentage of at bats than did
   the player with the highest number of homeruns.

2. Compare the top five players w.r.t. stolen bases (sb) to the top five players 
w.r.t. stolen as a percentage of in base appearances (ob).

    We already have our 2005 subset, but we will need to create a
    new "cleaned" version specific to this analysis.

    First we'll add a column for on base appearances (ob)

In [36]:
bb2005 = bb2005.assign(ob = bb2005['h'] \
               + bb2005['X2b'] \
               + bb2005['X3b'] \
               - bb2005['hr'] \
               + bb2005['bb'] \
               + bb2005['hbp'] )


In [37]:
print(bb2005.head())
print(bb2005.shape)


       Unnamed: 0         id  year  stint team  lg    g   ab   r   h  ...  \
21378       87533  sprinru01  2005      1  HOU  NL   62    0   0   0  ...   
21379       87610  micelda01  2005      1  COL  NL   19    0   0   0  ...   
21380       87634  sweenma01  2005      1  SDN  NL  135  221  31  65  ...   
21381       87636  hanseda01  2005      1  SEA  AL   60   75   5  13  ...   
21382       87641  willige02  2005      1  NYN  NL   39   30   9   7  ...   

        sb   cs  bb    so  ibb  hbp   sh   sf  gidp     ob  
21378  0.0  0.0   0   0.0  0.0  0.0  0.0  0.0   0.0    0.0  
21379  0.0  0.0   0   0.0  0.0  0.0  0.0  0.0   0.0    0.0  
21380  4.0  0.0  40  58.0  3.0  0.0  1.0  5.0   6.0  110.0  
21381  1.0  0.0   9  19.0  1.0  0.0  2.0  2.0   1.0   20.0  
21382  2.0  0.0   1   7.0  0.0  0.0  1.0  0.0   0.0    9.0  

[5 rows x 24 columns]
(160, 24)


In [38]:
bb2005_clean = bb2005.dropna(subset=['sb','ob'])
print(bb2005_clean.shape)


(160, 24)


    It appears there was no missing data (same number of rows before and after).
    Now I'll eliminate any rows with ob <= 0 and drop all columns
    except the three we need (id, sb, ob) to make the top 5 lists
    easier to work with

In [49]:
bb2005_clean = (bb2005[bb2005['ob']>0])[['id','sb','ob']]
print(bb2005_clean.shape)


(106, 3)


    It looks like we lost 54 players! That's a lot of no base appearances...
    Now we can proceed with # calculating the homeruns as a percentage of at 
    bats

In [50]:
bb2005_clean['sbpctob'] = bb2005_clean['sb'] / bb2005_clean['ob']
print(bb2005_clean.head())


              id   sb     ob   sbpctob
21380  sweenma01  4.0  110.0  0.036364
21381  hanseda01  1.0   20.0  0.050000
21382  willige02  2.0    9.0  0.222222
21383  harrile01  0.0   33.0  0.000000
21384  stairma01  1.0  188.0  0.005319


    The next step is to identify the top 5 players in terms 
    of number of home runs

In [51]:
top5sb = bb2005_clean.nlargest(5,'sb')
print(top5sb)


              id    sb     ob   sbpctob
21530  womacto01  27.0  104.0  0.259615
21489  vizquom01  24.0  244.0  0.098361
21504  loftoke01  22.0  175.0  0.125714
21491  lawtoma02  16.0  188.0  0.085106
21412  sandere02  14.0  107.0  0.130841


    ... and we can find the top 5 w.r.t. hr as % of ab

In [52]:
top5sbpctob = bb2005_clean.nlargest(5,'sbpctob')

In [53]:
print(top5sbpctob)


              id    sb     ob   sbpctob
21530  womacto01  27.0  104.0  0.259615
21382  willige02   2.0    9.0  0.222222
21402  williwo02   1.0    7.0  0.142857
21412  sandere02  14.0  107.0  0.130841
21504  loftoke01  22.0  175.0  0.125714


    Now we'll merge them to create a comparison table:
    First step is to replace the dataframe indexes so that the rows will align


In [55]:
top5sb.index = list(range(1,6))
top5sbpctob.index = list(range(1,6))


    Next, we concatenate the two dataframes. The keys will differentiate
    the columns between the absolute hr and hr as % ab groups


In [56]:
compare = pd.concat([top5sb, top5sbpctob], 
                    axis=1,
                    keys=['Absolute','Percentage'])
print(compare)


    Absolute                        Percentage                       
          id    sb     ob   sbpctob         id    sb     ob   sbpctob
1  womacto01  27.0  104.0  0.259615  womacto01  27.0  104.0  0.259615
2  vizquom01  24.0  244.0  0.098361  willige02   2.0    9.0  0.222222
3  loftoke01  22.0  175.0  0.125714  williwo02   1.0    7.0  0.142857
4  lawtoma02  16.0  188.0  0.085106  sandere02  14.0  107.0  0.130841
5  sandere02  14.0  107.0  0.130841  loftoke01  22.0  175.0  0.125714
